In [2]:
import dspy
import os
from dotenv import load_dotenv
from openai import OpenAI

/Users/pranitgunjal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")

In [4]:
client = OpenAI(api_key=API_KEY)

In [5]:
lm = dspy.LM("openai/o3", api_key=API_KEY, temperature=1, max_tokens=20_000)
dspy.configure(lm=lm)

In [6]:
class DataPromptSignature(dspy.Signature):
    """Signature for generating GPT-4 data-generation prompt."""
    data_description = dspy.InputField(desc="Description of the labeled, in-distribution data you want GPT-4 to generate")
    data_generation_prompt = dspy.OutputField(desc="Detailed prompt text to give GPT-4 for generating data")

class DataPromptGenerator(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate_prompt = dspy.Predict(signature=DataPromptSignature)

In [ ]:
# class DataPromptGenerator(dspy.Module):
#     def __init__(self):
#         super().__init__()
#         self.generate_prompt = dspy.Predict(
#             inputs=["data_description"],
#             outputs=["data_generation_prompt"],
#             instructions=(
#                 "Given a description of some type of labeled data, generate a detailed prompt "
#                 "that can be used with GPT-4 to create realistic in-distribution synthetic data. "
#                 "The prompt should be clear, specify the format, and provide examples if possible."
#             )
#         )

In [7]:
class DataPromptGenerator(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate_prompt = dspy.Predict(signature=DataPromptSignature)

In [10]:
prompt_generator = DataPromptGenerator()

desc = "A dataset of 10 realistic video game chat contents from a player in the DOTA 2 game, the content should be formed by combining all the messages from one player in a match and seperating the messages with a period. There will be some information about which words appear near each other. The labels should be for toxic and non-toxic players, with 1 representing non-toxic and -1 representing toxic players. The output should only be the text and label, no extra information"

res = prompt_generator.generate_prompt(data_description=desc)
print(res.data_generation_prompt)

You are generating a small, synthetic in-distribution dataset for a binary toxicity-detection task in the video-game domain (DOTA 2 match chat).  

Output requirements  
1. Produce EXACTLY 10 lines and nothing else (no headings, numbering, or explanations).  
2. Each line = one player’s combined chat messages for a single match, followed by a tab character, then the label.  
   • Chat messages: realistic in-game messages, concatenated together and separated with a single period (“.”).  
   • Label: “1”  =  non-toxic player, “-1”  =  toxic player.  
3. Mix of toxic and non-toxic: at least 4 of each class, remaining 2 can be either.  
4. Toxic content may include insults, profanity, sarcasm, blame, or flaming, but MUST NOT contain slurs or hateful language toward protected classes.  
5. Non-toxic content should sound friendly, strategic, or neutral.  
6. Keep each line between 25 and 60 words (count words, not characters).  
7. Do not wrap lines; each entry must be on one physical line.


In [11]:
print(res.data_generation_prompt)

You are generating a small, synthetic in-distribution dataset for a binary toxicity-detection task in the video-game domain (DOTA 2 match chat).  

Output requirements  
1. Produce EXACTLY 10 lines and nothing else (no headings, numbering, or explanations).  
2. Each line = one player’s combined chat messages for a single match, followed by a tab character, then the label.  
   • Chat messages: realistic in-game messages, concatenated together and separated with a single period (“.”).  
   • Label: “1”  =  non-toxic player, “-1”  =  toxic player.  
3. Mix of toxic and non-toxic: at least 4 of each class, remaining 2 can be either.  
4. Toxic content may include insults, profanity, sarcasm, blame, or flaming, but MUST NOT contain slurs or hateful language toward protected classes.  
5. Non-toxic content should sound friendly, strategic, or neutral.  
6. Keep each line between 25 and 60 words (count words, not characters).  
7. Do not wrap lines; each entry must be on one physical line.


In [12]:
res.data_generation_prompt

'You are generating a small, synthetic in-distribution dataset for a binary toxicity-detection task in the video-game domain (DOTA 2 match chat).  \n\nOutput requirements  \n1. Produce EXACTLY 10 lines and nothing else (no headings, numbering, or explanations).  \n2. Each line = one player’s combined chat messages for a single match, followed by a tab character, then the label.  \n   • Chat messages: realistic in-game messages, concatenated together and separated with a single period (“.”).  \n   • Label: “1”  =  non-toxic player, “-1”  =  toxic player.  \n3. Mix of toxic and non-toxic: at least 4 of each class, remaining 2 can be either.  \n4. Toxic content may include insults, profanity, sarcasm, blame, or flaming, but MUST NOT contain slurs or hateful language toward protected classes.  \n5. Non-toxic content should sound friendly, strategic, or neutral.  \n6. Keep each line between 25 and 60 words (count words, not characters).  \n7. Do not wrap lines; each entry must be on one phy

In [18]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": res.data_generation_prompt}
    ]
)

In [12]:
print(response.choices[0].message.content)

Here's a dataset with 10 examples of DOTA 2 chat content, labeled as non-toxic (1) or toxic (-1):

1. Non-Toxic (1):
   - "Nice last hit, Juggernaut! Keep it up and we'll outfarm them!"

2. Non-Toxic (1):
   - "Can we group up for a smoke gank? I think we can catch them off guard."

3. Non-Toxic (1):
   - "Great warding, thanks for the vision support. Makes it much easier to roam."

4. Non-Toxic (1):
   - "No worries about the early deaths, let's focus on farming and come back stronger in mid-game!"

5. Non-Toxic (1):
   - "Good job everyone! That was a tough fight but we played it well. Let's keep the momentum."

6. Toxic (-1):
   - "Are you blind, or just stupid? How did you miss that stun?"

7. Toxic (-1):
   - "GG, this team is hopeless. I'm just gonna AFK in base now."

8. Toxic (-1):
   - "Wow, our mid is feeding like it's a buffet. Report this noob."

9. Toxic (-1):
   - "Can you stop pretending like you know how to play? You're ruining the game."

10. Toxic (-1):
    - "Nice, a

In [19]:
print(response.choices[0].message.content)

"chat","label"  
"gg wp, team! Nice work on those last fights!",1  
"Why are you still in mid? You're useless 😂",1  
"pls remember to ward jungle, we're getting ganked too much",1  
"wtf are you doing?? uninstall pls",–1  
"awesome rotations, support! keep it up!",1  
"ez game, you guys were trash 🤣",–1  
"good hustle, we'll get them next time!",1  
"stop feeding, you noob!",–1  
"that's the worst build I've ever seen 🤦",–1  
"thanks for the carry, team! such a fun match!",1  
